# VIQA · Hỏi đáp trích xuất tiếng Việt (Jaccard mức âm tiết) - KHÔNG dùng trọng số pretrained

Chỉ có 1190 mẫu tiếng Việt, quá ít để train một mô hình đọc hiểu nơ-ron từ đầu, nên dùng hướng **sinh span ứng viên rồi xếp hạng**:
1. Tính IDF trên các context, chọn `K_SENT` câu giống câu hỏi nhất (tổng IDF của từ câu hỏi xuất hiện trong câu).
2. Liệt kê mọi span dài 1-`LMAX` âm tiết trong các câu đó (không bắt đầu/kết thúc bằng dấu câu, không trùng hết với câu hỏi).
3. Đặc trưng thủ công cho từng span: khoảng cách tới từ của câu hỏi, số từ câu hỏi quanh span, loại câu hỏi ("bao nhiêu", "ai", "khi nào"...),
   span có chữ số / chữ hoa, độ dài, hạng của câu...
4. **XGBoost** chấm điểm span (so sánh: hồi quy Jaccard / **XGBRanker** pairwise theo từng câu hỏi), chọn span điểm cao nhất.
5. Dùng thêm 3000 mẫu **SQuAD tiếng Anh** (dữ liệu được cấp): đặc trưng cấu trúc không phụ thuộc ngôn ngữ, từ hỏi tiếng Anh
   được ánh xạ sang loại câu hỏi tiếng Việt (how many: bao nhiêu, when: khi nào, who: ai...).

Chạy hoàn toàn trên CPU, ~5-8 phút. Thư viện: `numpy`, `pandas`, `xgboost`, `json`, `re`, `collections`, `math`.

In [ ]:
import json, re, math, time, random
from collections import Counter
import numpy as np
import pandas as pd
from xgboost import XGBRegressor, XGBRanker

DATA = '.'
K_SENT, LMAX, WIN = 2, 10, 6
SEED = 42
random.seed(SEED); np.random.seed(SEED)
T0 = time.time()
load = lambda f: json.load(open(f'{DATA}/{f}.json', encoding='utf-8'))
train_vi, val_vi, test = load('train_vi'), load('validation_vi'), load('test')
print(len(train_vi), len(val_vi), len(test))

_PUNCT = re.compile(r'[^\w\s]', re.UNICODE)
norm_words = lambda s: _PUNCT.sub(' ', s.lower()).split()


def jaccard(a, b):
    a, b = set(norm_words(a)), set(norm_words(b))
    return len(a & b) / len(a | b) if (a | b) else 1.0


def score(preds, data):
    g = [r['answers'][0]['text'] for r in data]
    p = [preds[r['id']] for r in data]
    return {'jaccard': round(np.mean([jaccard(x, y) for x, y in zip(g, p)]), 4),
            'em': round(np.mean([norm_words(x) == norm_words(y) for x, y in zip(g, p)]), 4)}

In [ ]:
lens = [len(norm_words(r['answers'][0]['text'])) for r in train_vi + val_vi]
print('độ dài đáp án (âm tiết): trung vị', np.median(lens), '| phân vị 90:', np.percentile(lens, 90), '| tỉ lệ ≤ LMAX:', round(np.mean(np.array(lens) <= LMAX), 3))

## 1. Tách câu, tách token (giữ vị trí ký tự), IDF

In [ ]:
TOK = re.compile(r'\d+(?:[.,]\d+)*|\w+|[^\w\s]')
SENT_END = re.compile(r'(?<=[.!?])\s+')


def sentences(ctx):
    """Danh sách câu, mỗi câu là list token (text, start, end) với vị trí ký tự trong context."""
    out, pos = [], 0
    for part in SENT_END.split(ctx):
        st = ctx.index(part, pos); pos = st + len(part)
        toks = [(m.group(), st + m.start(), st + m.end()) for m in TOK.finditer(part)]
        if toks: out.append(toks)
    return out


docs = {}
for r in train_vi + val_vi + test:
    docs.setdefault(r['context'], None)
df = Counter()
n_docs = 0
for ctx in docs:
    for s in sentences(ctx):
        df.update({t.lower() for t, _, _ in s}); n_docs += 1
idf = lambda w: math.log((n_docs + 1) / (df.get(w, 0) + 1)) + 1
print('số câu dùng tính IDF:', n_docs)

## 2. Sinh span ứng viên + đặc trưng

In [ ]:
QTYPES = ['bao nhiêu', 'mấy', 'khi nào', 'năm nào', 'bao giờ', 'thời gian', 'ai', 'ở đâu', 'đâu', 'nơi',
          'gì', 'nào', 'tại sao', 'vì sao', 'như thế nào', 'bao lâu', 'là gì', 'tên']
FEATS = (['sent_rank', 'sent_score', 'sent_rel', 'sent_pos', 'L', 'nchar', 'frac_q', 'n_punct', 'n_digit', 'n_cap',
          'first_cap', 'last_digit', 'q_win', 'idf_win', 'dist_left', 'dist_right', 'prev_q', 'next_q', 'has_nam']
         + [f'qt_{q}' for q in QTYPES])


# từ hỏi tiếng Anh ánh xạ sang loại câu hỏi tiếng Việt, để dùng thêm SQuAD tiếng Anh (đặc trưng cấu trúc không phụ thuộc ngôn ngữ)
EN2VI = {'how many': 'bao nhiêu', 'how much': 'bao nhiêu', 'when': 'khi nào', 'what year': 'năm nào', 'what time': 'thời gian',
         'who': 'ai', 'whom': 'ai', 'whose': 'ai', 'where': 'ở đâu', 'what': 'gì', 'which': 'nào', 'why': 'tại sao',
         'how long': 'bao lâu', 'how': 'như thế nào', 'name': 'tên'}
has_word = lambda pat, q: re.search(rf'(?<!\w){pat}(?!\w)', q) is not None    # khớp nguyên từ ("ai" không khớp "hai")


def candidates(r, with_label, lang='vi'):
    q = r['question'].lower()
    qset = {t.lower() for t in TOK.findall(r['question']) if re.match(r'\w', t)}
    if lang == 'vi':
        qflags = [int(has_word(qt, q)) for qt in QTYPES]
    else:
        hit = {vi for en, vi in EN2VI.items() if has_word(en, q)}
        qflags = [int(qt in hit) for qt in QTYPES]
    sents = sentences(r['context'])
    sc = [sum(idf(w) for w in qset & {t.lower() for t, _, _ in s}) for s in sents]
    order = np.argsort(sc)[::-1][:K_SENT]
    best_sc = max(sc) if sc and max(sc) > 0 else 1.0
    rows, spans = [], []
    gold = r['answers'][0]['text'] if with_label else None
    for rank, si in enumerate(order):
        s = sents[si]; n = len(s)
        low = [t.lower() for t, _, _ in s]
        isw = np.array([bool(re.match(r'\w', t)) for t, _, _ in s])
        isq = np.array([w in qset for w in low])
        dig = np.array([any(ch.isdigit() for ch in t) for t, _, _ in s])
        cap = np.array([t[:1].isupper() for t, _, _ in s])
        wq = np.array([idf(w) if w in qset else 0.0 for w in low])
        cw, cq, cd, cc, ci = (np.concatenate([[0], np.cumsum(a)]) for a in (isw, isq, dig, cap, wq))
        qpos = np.where(isq)[0]
        for i in range(n):
            if not isw[i]: continue
            for j in range(i, min(n, i + LMAX)):
                if not isw[j]: continue
                L = j - i + 1
                nq_in = cq[j + 1] - cq[i]
                if nq_in == L: continue
                a, b = max(0, i - WIN), min(n, j + 1 + WIN)
                left = qpos[qpos < i]; right = qpos[qpos > j]
                rows.append([rank, sc[si], sc[si] / best_sc, si / max(1, len(sents) - 1), L, s[j][2] - s[i][1],
                             nq_in / L, L - (cw[j + 1] - cw[i]), cd[j + 1] - cd[i], cc[j + 1] - cc[i],
                             int(cap[i]), int(dig[j]), (cq[i] - cq[a]) + (cq[b] - cq[j + 1]), (ci[i] - ci[a]) + (ci[b] - ci[j + 1]),
                             i - left[-1] if len(left) else 99, right[0] - j if len(right) else 99,
                             int(i > 0 and isq[i - 1]), int(j + 1 < n and isq[j + 1]),
                             int(any(w in ('năm', 'year', 'years') for w in low[i:j + 1]))] + qflags)
                spans.append(r['context'][s[i][1]:s[j][2]])
    y = [jaccard(sp, gold) for sp in spans] if with_label else None
    return rows, spans, y


def build(data, with_label, lang='vi'):
    X, Y, grp, spans = [], [], [], []
    for k, r in enumerate(data):
        rows, sp, y = candidates(r, with_label, lang)
        X += rows; spans += sp; grp += [k] * len(rows)
        if with_label: Y += y
    return np.array(X, np.float32), (np.array(Y, np.float32) if with_label else None), np.array(grp), spans


Xtr, Ytr, Gtr, Str = build(train_vi, True)
Xva, Yva, Gva, Sva = build(val_vi, True)
print(f'ứng viên train {len(Xtr):,} · val {len(Xva):,} · {(time.time() - T0) / 60:.1f} phút')
oracle = pd.Series(Yva).groupby(Gva).max()
print('trần (span ứng viên tốt nhất) trên val: Jaccard', round(oracle.reindex(range(len(val_vi)), fill_value=0).mean(), 4))

## 3. XGBoost dự đoán Jaccard của span, chọn span tốt nhất mỗi câu hỏi

In [ ]:
def pick(model, X, G, spans, data):
    p = model.predict(X)
    best = pd.DataFrame({'g': G, 'p': p}).groupby('g').p.idxmax()
    return {data[g]['id']: spans[i].strip(' .,;:!?"\'') for g, i in best.items()} | \
           {r['id']: '' for k, r in enumerate(data) if k not in best.index}


params = dict(n_estimators=400, max_depth=7, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8,
              tree_method='hist', random_state=SEED, n_jobs=-1)
rel = lambda y: np.round(y * 4).astype(int)       # nhãn mức liên quan 0..4 cho bài toán xếp hạng

# thêm một phần SQuAD tiếng Anh (dữ liệu được cấp): chỉ dùng đặc trưng cấu trúc, từ hỏi đã ánh xạ sang loại câu hỏi tiếng Việt
N_EN = 3000
train_en = load('train_en')
en_sub = [train_en[i] for i in np.random.RandomState(SEED).choice(len(train_en), N_EN, replace=False)]
Xen, Yen, Gen, _ = build(en_sub, True, lang='en')
print(f'ứng viên tiếng Anh {len(Xen):,} · {(time.time() - T0) / 60:.1f} phút')

variants = {
    'hồi quy (train_vi)':          (lambda: XGBRegressor(**params).fit(Xtr, Ytr)),
    'xếp hạng (train_vi)':         (lambda: XGBRanker(objective='rank:pairwise', **params).fit(Xtr, rel(Ytr), qid=Gtr)),
    'xếp hạng (train_vi + en)':    (lambda: XGBRanker(objective='rank:pairwise', **params).fit(
        np.concatenate([Xtr, Xen]), rel(np.concatenate([Ytr, Yen])), qid=np.concatenate([Gtr, Gen + Gtr.max() + 1]))),
}
res_v, models = [], {}
for name, fit in variants.items():
    models[name] = fit()
    res_v.append({'cách': name, **score(pick(models[name], Xva, Gva, Sva, val_vi), val_vi)})
    print(res_v[-1], f'| {(time.time() - T0) / 60:.1f} phút')
BEST = max(res_v, key=lambda r: r['jaccard'])['cách']
model = models[BEST]
p_val = pick(model, Xva, Gva, Sva, val_vi)
print('chọn:', BEST)

imp = pd.Series(model.feature_importances_, index=FEATS).sort_values(ascending=False)
print('đặc trưng quan trọng nhất:', imp.head(8).round(3).to_dict())

So sánh với baseline từ vựng của `starter.py` (chọn câu trùng nhiều từ nhất, lấy 5 từ đầu không thuộc câu hỏi):

In [ ]:
def starter_predict(r):
    words = lambda s: re.findall(r'\w+', s.lower())
    q = set(words(r['question']))
    sents = [s for s in re.split(r'(?<=[.!?])\s+', r['context']) if s.strip()]
    best = max(sents, key=lambda s: len(q & set(words(s))))
    keep = [t for t in best.replace(',', ' ,').split() if t.lower().strip('.,;:!?()"') not in q]
    return ' '.join(keep[:5]).strip('.,;:!?')


print('starter val:', score({r['id']: starter_predict(r) for r in val_vi}, val_vi))
err = pd.DataFrame([{'question': r['question'], 'gold': r['answers'][0]['text'], 'pred': p_val[r['id']],
                     'jac': jaccard(r['answers'][0]['text'], p_val[r['id']])} for r in val_vi])
err.sort_values('jac').head(10)

## 4. Bản nộp: train lại trên train_vi + validation, dự đoán test

In [ ]:
Xte, _, Gte, Ste = build(test, False)
X_all, Y_all, G_all = np.concatenate([Xtr, Xva]), np.concatenate([Ytr, Yva]), np.concatenate([Gtr, Gva + Gtr.max() + 1])
if 'en' in BEST:
    X_all, Y_all = np.concatenate([X_all, Xen]), np.concatenate([Y_all, Yen])
    G_all = np.concatenate([G_all, Gen + G_all.max() + 1])
if BEST.startswith('hồi quy'):
    model_full = XGBRegressor(**params).fit(X_all, Y_all)
else:
    model_full = XGBRanker(objective='rank:pairwise', **params).fit(X_all, rel(Y_all), qid=G_all)
p_test = pick(model_full, Xte, Gte, Ste, test)
sub = pd.DataFrame({'id': [r['id'] for r in test], 'answer': [p_test[r['id']] for r in test]})
assert len(sub) == len(test) and sub.id.is_unique
sub.to_csv('submission.csv', index=False, encoding='utf-8')
model_full.save_model('viqa_xgb.json')
print(f'đã ghi submission.csv sau {(time.time() - T0) / 60:.1f} phút'); sub.head(10)